# Extract dates from historical notes

Configure the input paths in the setup cell, then run the cells in order. See the project README for input schemas and model setup.


In [ ]:
from pathlib import Path
import sys

# Locate the project when Jupyter starts in a notebook subfolder.
project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "smmnlp" / "__init__.py").is_file()),
    None,
)
if project_root is None:
    raise RuntimeError("Start Jupyter inside SMMnlp_v2 or set its working directory there.")
sys.path.insert(0, str(project_root))


In [ ]:
import logging

import pandas as pd

from smmnlp.config import ModelConfig
from smmnlp.inference import generate_responses
from smmnlp.modeling import LocalLLM
from smmnlp.paths import ProjectPaths
from smmnlp.parsing import parse_history_for_extraction
from smmnlp.preprocessing import add_parsed_labels
from smmnlp.prompts import Task

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
paths = ProjectPaths.from_environment()
# Override paths here, or set SMMNLP_DATA_ROOT and SMMNLP_MODEL_PATH.
from smmnlp.preprocessing import add_date_extractions


## Patient-level hx sMM Date extraction 1. Find patients LLM labeled as positive for SMM (n=200, n=83) 2. Identify the first positive SMM encournter 3. Identify the first positive SMM encournter LLM labeled as hx (n=72, n=34) 4.1 Extract the date of dx 4.2 Determine whether the extracted date of dx needs human review


In [ ]:
readfile_name = paths.data("Update/Data/Testing/EndToEnd_testing_SMMyesdataset.json")
source_notes = pd.read_json(readfile_name)

In [ ]:
len(source_notes)

In [ ]:
source_notes.head(5)

In [ ]:
llmdata = paths.data("Update/ExtraLLMRun/EndToEnd_testing_SMMyesdataset_hx.xlsx")
history_outputs = pd.read_excel(llmdata)

In [ ]:
len(history_outputs)

In [ ]:
history_outputs.head(5)

## POST PROCESSING


In [ ]:
llm_columns = ["HxNew_LLM_Binary"]
history_outputs = add_parsed_labels(history_outputs, llm_columns, parse_history_for_extraction)

In [ ]:
note_columns = source_notes[["PatientSSN", "NoteDate", "NoteText"]]

In [ ]:
prediction_columns = history_outputs[["HxNew_LLM_Binary", "post_HxNew_LLM_Binary"]]

In [ ]:
combined_notes = pd.concat(
    [note_columns.reset_index(drop=True), prediction_columns.reset_index(drop=True)], axis=1
)

In [ ]:
len(combined_notes)

## Identify the hx SMM dx


In [ ]:
outcome3 = combined_notes[combined_notes["post_HxNew_LLM_Binary"] == "Hx"]

In [ ]:
len(outcome3)

In [ ]:
generator = LocalLLM.from_pretrained(ModelConfig(paths.model_path))

In [ ]:
notes = outcome3
note_count = len(notes)

In [ ]:
notes.head(5)

In [ ]:
responses = generate_responses(notes["NoteText"].tolist(), generator, Task.DATE)

In [ ]:
responses

In [ ]:
notes["rawoutput"] = responses

In [ ]:
notes = add_date_extractions(notes)


In [ ]:
notes.head(5)

In [ ]:
savefile_name = paths.data("Update/ExtraLLMRun/EndToEnd_testing_dxdate_human.xlsx")
notes.to_excel(savefile_name, engine="xlsxwriter", sheet_name="Sheet1", index=False)